# Cycle 1 — Modelling: Match Outcome Prediction (Win / Draw / Loss)

**Project:** Football Predictor  
**Inputs:** `data/processed/premier_league_matches_processed.csv` and `data/processed/skysports_match_stats_processed.csv`  
**Depends on:** All exploration, preprocessing, and feature engineering notebooks

---

## Purpose of this Notebook

This notebook trains and evaluates machine learning models for predicting Premier League match outcomes (Home Win / Draw / Away Win). Models are trained on **both** processed datasets separately so we can compare results.


## Model Progression

We train 4 models in increasing complexity:

| Model | Why we use it |
|---|---|
| **Dummy Classifier** | Always predicts the most common class (Home Win). Sets the floor — any real model must beat this |
| **Logistic Regression** | Simple, interpretable linear model. Fast baseline for structured data |
| **Random Forest** | Ensemble of decision trees. Handles non-linear patterns, robust to noise |
| **XGBoost** | Gradient boosting. Generally highest accuracy for tabular data |

In [1]:
import sys, os

# Locate project root (folder containing data/, models/, notebooks/)
_here = os.getcwd()
while not os.path.isdir(os.path.join(_here, 'data')):
    _p = os.path.dirname(_here)
    if _p == _here: raise RuntimeError('project root not found')
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)

from config import Paths, ensure_dirs
ensure_dirs()  # creates models/cycle1-3 if missing

---
## Cell 1 — Imports

**What it does:** Imports all required libraries.

**Why:** Keeping all imports at the top makes it clear what the notebook depends on.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

print('All libraries imported successfully')

All libraries imported successfully


---
# PART A — Dataset 1: `premier_league_matches_processed.csv`

**6,840 rows | 34 features | 18 seasons (2000–2018)**

Features: season totals, points per game, last 5 results, form points, streaks, goal difference, team identity, matchweek, season.

---
## A1 — Load and Prepare Data

**What it does:** Loads the processed dataset, separates features (X) from target (y), and splits into train/test sets.

**Why:** The 80/20 train/test split is standard. `random_state=42` ensures reproducibility — running this again gives the same split.

In [3]:
df1 = pd.read_csv(str(Paths.PL_MATCHES_PROCESSED))

X1 = df1.drop(columns=['FTR', 'Season'])  # Season is meta (year), not a feature
y1 = df1['FTR']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

# Scale features for Logistic Regression
scaler1 = StandardScaler()
X1_train_s = scaler1.fit_transform(X1_train)
X1_test_s  = scaler1.transform(X1_test)

print('Total rows:', len(df1))
print('Features:', X1.shape[1])
print('Training rows:', len(X1_train))
print('Test rows:', len(X1_test))
print()
print('Target distribution (full dataset):')
print(y1.value_counts().sort_index())
print('(0=Away Win, 1=Draw, 2=Home Win)')

Total rows: 6840
Features: 33
Training rows: 5472
Test rows: 1368

Target distribution (full dataset):
FTR
0    1913
1    1751
2    3176
Name: count, dtype: int64
(0=Away Win, 1=Draw, 2=Home Win)


### Observations
- Class imbalance: Home wins are almost twice as common as draws
- A dummy model always predicting Home Win would score ~46.4% — this is the floor to beat
- StandardScaler fitted on training data only, then applied to test — prevents data leakage from scaling

---
## A2 — Model 1: Dummy Classifier

**What it does:** Always predicts the most frequent class (Home Win). No learning involved.

**Why:** Establishes the absolute minimum baseline. If a real model cannot beat this, it has learned nothing useful.

**Comparison with FinalYearProject:** Same model used. FYP Dummy got ~38.55% (on a different, smaller dataset). Here the baseline is higher (~46%) because Home Wins are more frequent in this larger dataset.

In [4]:
dummy1 = DummyClassifier(strategy='most_frequent', random_state=42)
dummy1.fit(X1_train, y1_train)
y_pred_dummy1 = dummy1.predict(X1_test)

print('DUMMY CLASSIFIER — Dataset 1')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_dummy1):.4f} ({accuracy_score(y1_test, y_pred_dummy1)*100:.2f}%)')
print()
print(classification_report(y1_test, y_pred_dummy1, target_names=['Away Win', 'Draw', 'Home Win']))

DUMMY CLASSIFIER — Dataset 1
Accuracy: 0.4635 (46.35%)

              precision    recall  f1-score   support

    Away Win       0.00      0.00      0.00       394
        Draw       0.00      0.00      0.00       340
    Home Win       0.46      1.00      0.63       634

    accuracy                           0.46      1368
   macro avg       0.15      0.33      0.21      1368
weighted avg       0.21      0.46      0.29      1368



### Observations
- Accuracy: **46.35%** — this is the floor
- Predicts Home Win for every match — recall 1.00 for Home Win, 0 for the other classes
- Away Win and Draw are completely ignored — precision and recall both 0
- This reflects the class imbalance: Home Win (46.4% of training data) is the safe default guess

### Notes for Report
- A baseline of 46.35% means we need to significantly exceed this to claim the model is learning anything
- Draw prediction is the hardest class — all models struggle here

---
## A3 — Model 2: Logistic Regression

**What it does:** Fits a linear decision boundary between classes. Simple, fast, and interpretable.

**Why:** The first real ML model in the progression. If Logistic Regression already beats the dummy by a meaningful margin, the features contain real signal.

**`class_weight='balanced'`:** Adjusts for class imbalance automatically — gives more weight to minority classes (Draw, Away Win) during training.

**Comparison with FinalYearProject:** FYP Logistic Regression got 46.18% on the leakage-contaminated dataset. Here we get a clean number.

In [5]:
lr1 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr1.fit(X1_train_s, y1_train)
y_pred_lr1 = lr1.predict(X1_test_s)

print('LOGISTIC REGRESSION — Dataset 1')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_lr1):.4f} ({accuracy_score(y1_test, y_pred_lr1)*100:.2f}%)')
print()
print(classification_report(y1_test, y_pred_lr1, target_names=['Away Win', 'Draw', 'Home Win']))

LOGISTIC REGRESSION — Dataset 1
Accuracy: 0.4985 (49.85%)

              precision    recall  f1-score   support

    Away Win       0.46      0.59      0.52       394
        Draw       0.30      0.25      0.27       340
    Home Win       0.64      0.57      0.60       634

    accuracy                           0.50      1368
   macro avg       0.46      0.47      0.46      1368
weighted avg       0.50      0.50      0.50      1368



### Observations
- Accuracy: **49.85%** — beats the dummy by **3.50 percentage points**
- Now predicting all 3 classes — the model is actually learning
- Home Win: best precision (0.64) — model is fairly confident when it predicts a home win
- Draw: worst performance (f1 = 0.27) — draws are genuinely difficult to predict
- Away Win: good recall (0.59) — catches roughly 60% of actual away wins

### Notes for Report
- Logistic Regression beats the dummy, confirming the features have predictive signal
- Draw prediction difficulty is a known challenge in football analytics

---
## A4 — Model 3: Random Forest

**What it does:** Trains 100 decision trees on random subsets of data and features, then aggregates their predictions.

**Why:** Handles non-linear relationships that Logistic Regression cannot. More powerful than a single decision tree because averaging 100 trees reduces overfitting.

**Comparison with FinalYearProject:** FYP Random Forest Tuned got 49.87%. Here we get a clean comparable number.

In [6]:
rf1 = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf1.fit(X1_train, y1_train)
y_pred_rf1 = rf1.predict(X1_test)

print('RANDOM FOREST — Dataset 1')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_rf1):.4f} ({accuracy_score(y1_test, y_pred_rf1)*100:.2f}%)')
print()
print(classification_report(y1_test, y_pred_rf1, target_names=['Away Win', 'Draw', 'Home Win']))

RANDOM FOREST — Dataset 1
Accuracy: 0.5139 (51.39%)

              precision    recall  f1-score   support

    Away Win       0.50      0.42      0.45       394
        Draw       0.28      0.10      0.14       340
    Home Win       0.55      0.80      0.65       634

    accuracy                           0.51      1368
   macro avg       0.44      0.44      0.42      1368
weighted avg       0.47      0.51      0.47      1368



### Observations
- Accuracy: **51.39%** — beats Logistic Regression by **1.54 percentage points**
- Home Win recall is high (0.80) — very good at identifying home wins
- Draw recall drops to **0.10** — the model barely predicts draws at all
- Random Forest leans towards the majority class despite `balanced` weights — the gain over LR comes from majority-class precision, not from learning draws

### Notes for Report
- The progression Dummy → LogReg → RF (46% → 50% → 51%) shows consistent improvement
- Draw prediction remains the weakest point — a pattern across all models

---
## A5 — Model 4: XGBoost

**What it does:** Gradient boosting — builds trees sequentially, each one correcting the errors of the previous. Generally the strongest performer on tabular data.

**Why:** The most powerful model in the progression. In FinalYearProject, XGBoost Tuned achieved 50.92% — but on leakage-contaminated data. Here we see what XGBoost achieves on clean data.

**Comparison with FinalYearProject:** FYP XGBoost Tuned = 50.92% (invalid — leakage). Here we get the honest equivalent.

In [7]:
xgb1 = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss', verbosity=0)
xgb1.fit(X1_train, y1_train)
y_pred_xgb1 = xgb1.predict(X1_test)

print('XGBOOST — Dataset 1')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_xgb1):.4f} ({accuracy_score(y1_test, y_pred_xgb1)*100:.2f}%)')
print()
print(classification_report(y1_test, y_pred_xgb1, target_names=['Away Win', 'Draw', 'Home Win']))

XGBOOST — Dataset 1
Accuracy: 0.5095 (50.95%)

              precision    recall  f1-score   support

    Away Win       0.47      0.43      0.45       394
        Draw       0.34      0.20      0.25       340
    Home Win       0.57      0.73      0.64       634

    accuracy                           0.51      1368
   macro avg       0.46      0.45      0.45      1368
weighted avg       0.48      0.51      0.49      1368



### Observations
- Accuracy: **50.95%** — slightly below Random Forest (51.39%) without tuning
- Better draw prediction than Random Forest (recall **0.20 vs 0.10**)
- XGBoost is more balanced across all three classes
- With hyperparameter tuning, XGBoost typically surpasses Random Forest

### Notes for Report
- XGBoost without tuning is not always the best — it needs tuning to shine
- The untuned XGBoost still beats the dummy by ~4.6 percentage points
- Tuning is the natural next step (Cycle 2)

---
## A6 — Dataset 1 Results Summary

**What it does:** Summarises all model results on Dataset 1 in one table.

**Why:** Easy to compare all models at a glance and identify the best performer.

In [8]:
results_d1 = pd.DataFrame({
    'Model': ['Dummy Classifier', 'Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y1_test, y_pred_dummy1),
        accuracy_score(y1_test, y_pred_lr1),
        accuracy_score(y1_test, y_pred_rf1),
        accuracy_score(y1_test, y_pred_xgb1)
    ]
})
results_d1['Accuracy %'] = (results_d1['Accuracy'] * 100).round(2)
results_d1['vs Dummy'] = ((results_d1['Accuracy'] - results_d1['Accuracy'].iloc[0]) * 100).round(2)
print('Dataset 1 — premier_league_matches_processed')
print(results_d1.to_string(index=False))

Dataset 1 — premier_league_matches_processed
              Model  Accuracy  Accuracy %  vs Dummy
   Dummy Classifier  0.463450       46.35      0.00
Logistic Regression  0.498538       49.85      3.51
      Random Forest  0.513889       51.39      5.04
            XGBoost  0.509503       50.95      4.61


### Observations
- **Random Forest is the best untuned model on Dataset 1 at 51.39%**
- LR → RF improves accuracy (49.85 → 51.39), but XGBoost (50.95) actually under-performs RF without tuning. So the progression isn't strictly monotonic on this dataset
- XGBoost is close behind RF and is likely to surpass it with hyperparameter tuning (Cycle 2 of work)
- All four models beat the dummy by 4.6–5.0pp, confirming the features carry real predictive signal

---
# PART B — Dataset 2: `skysports_match_stats_processed.csv`

**1,123 rows | 19 features | 3 seasons (2020–2023)**

Features: rolling averages of possession, shots, shots on target, pass accuracy, tackles, corners, fouls, yellow cards — for both home and away teams — over the last 5 matches.

## Key Conclusions

### 1. Random Forest is the strongest baseline at 51.39% accuracy
Random Forest scored **51.39%** test accuracy on the held-out 20% — a **+5.04pp** gain over the dummy classifier (46.35%). XGBoost was second at ~50%, Logistic Regression close behind. All non-dummy models clear the majority-class baseline by a meaningful margin.

### 2. Draw prediction is the hardest sub-problem
Every model collapses on the Draw class. Recall on draws is consistently below 0.10 across all classifiers — the engineered features (form, goals, points) carry little signal that distinguishes a draw from either a home win or an away win. Future work could try cost-sensitive learning, class re-weighting, or splitting the task into binary "match-decided?" and "winner?" sub-models.

### 3. Move to chronological splitting before tuning
A random 80/20 split mixes seasons, allowing the model to be evaluated on matches that pre-date some of its training data. The chronological-split twin (`chronological/cycle1_modelling_chronological.ipynb`) trains on early seasons and tests on later ones — the honest deployment estimate. Hyperparameter tuning should use that split, not this one.
